# Consolidation pipeline

In [1]:
# Import Packages 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
%matplotlib inline
import seaborn as sns

In [2]:
# Create timestamp and author
from datetime import datetime 
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print("Current Time:{}".format(time_now))
print("Dorthy Tran")

Current Time:11_24_2026_0427PM
Dorthy Tran


In [3]:
# Import all CSV files
us = pd.read_csv("us_region_sales-1.csv")
eu = pd.read_csv("eu_region_sales.csv")
apac = pd.read_csv("apac_region_sales-1.csv")
rates = pd.read_csv("exchange_rates (1).csv")
products = pd.read_csv("product_master (1).csv")
sales = pd.read_csv("Consolidated Sales.csv")

In [4]:
# Standardize transaction dates
us["Date"] = pd.to_datetime(us["Sale_Date"], format="%m/%d/%Y", errors="coerce")
eu["Date"] = pd.to_datetime(eu["date"], format="%Y-%m-%d", errors="coerce")
apac["Date"] = pd.to_datetime(apac["txn_date_ddmmyyyy"], format="%d-%m-%Y", errors="coerce")
rates["Date"] = pd.to_datetime(rates["rate_date"], format="%Y-%m-%d", errors="coerce")

In [5]:
# Check for invalid dates
print("US:", us["Date"].isna().sum())
print("EU:", eu["Date"].isna().sum())
print("APAC:", apac["Date"].isna().sum())
print("Exchange Rates:", rates["Date"].isna().sum())

US: 4
EU: 3
APAC: 3
Exchange Rates: 0


In [6]:
print(sales.columns.tolist())

['City', 'Qty', 'Currency', 'Region', 'Index', 'Order ID', 'Raw Date', 'Clean Date', 'Product Name', 'Raw Local Amount', 'Local Unit Price', 'Tax Percentage']


In [7]:
# Display invalid US dates
us[us["Date"].isna()][["Order_No", "Sale_Date"]].rename(columns={"Order_No": "Order No", "Sale_Date": "Sale Date"})

,Order No,Sale Date
130,US-100203,17/02/2026
336,US-100527,13/05/2026
341,US-100534,14/05/2026
618,US-100967,20/08/2026


In [8]:
# [9] Display invalid EU dates
eu[eu["Date"].isna()][["order_ref", "date"]].rename(columns={"order_ref": "Order No", "date": "Sale Date"})

,Order No,Sale Date
55,EU-200090,30.01.2026
208,EU-200336,12.04.2026
464,EU-200743,31.07.2026


In [9]:
# Display invalid APAC dates
apac[apac["Date"].isna()][["txn_ref", "txn_date_ddmmyyyy"]].rename(columns={"txn_ref": "Order No", "txn_date_ddmmyyyy": "Sale Date"})

,Order No,Sale Date
104,AP-300159,2026-02-26
209,AP-300341,2026-04-29
434,AP-300670,2026-08-23


In [10]:
# Calculate original local revenue
us["Local_Revenue"] = us["Qty_Sold"] * us["Unit_Price_USD"]
eu["Local_Revenue"] = eu["line_total_eur_incl_vat"]
apac["Local_Revenue"] = apac["qty"] * apac["unit_price_local"]

In [11]:
# Identify duplicate exchange rates
rates[rates.duplicated(subset=["Date", "currency"], keep=False)][["rate_date", "currency", "usd_per_unit"]].rename(
    columns={"rate_date": "Rate Date", "currency": "Currency", "usd_per_unit": "USD Per Unit"})

,Rate Date,Currency,USD Per Unit
195,2026-02-19,EUR,1.077520
196,2026-02-19,EUR,1.206820
859,2026-08-05,JPY,0.007459
860,2026-08-05,JPY,0.006660


In [12]:
# Keep the first exchange rate for each date and currency
original_rates = rates.drop_duplicates(subset=["Date", "currency"], keep="first")

In [13]:
# Assign EUR currency to European sales
eu["currency"] = "EUR"

# Match European exchange rates
eu_usd = eu.merge(
    original_rates[["Date", "currency", "usd_per_unit"]],
    on=["Date", "currency"],
    how="left"
)

# Match APAC exchange rates
apac_usd = apac.merge(
    original_rates[["Date", "currency", "usd_per_unit"]],
    on=["Date", "currency"],
    how="left"
)

In [14]:
# Convert local revenue to USD
us["In_USD"] = us["Local_Revenue"]
eu_usd["In_USD"] = eu_usd["Local_Revenue"] * eu_usd["usd_per_unit"]
apac_usd["In_USD"] = apac_usd["Local_Revenue"] * apac_usd["usd_per_unit"]

In [15]:
# Count missing exchange rates for valid dates
print("EU:", eu_usd["usd_per_unit"].isna().mul(eu_usd["Date"].notna()).sum())
print("APAC:", apac_usd["usd_per_unit"].isna().mul(apac_usd["Date"].notna()).sum())

EU: 10
APAC: 9


In [16]:
# Calculate original USD revenue by region
us_total = us.loc[us["Date"].notna(), "In_USD"].sum()
eu_total = eu_usd.loc[eu_usd["Date"].notna(), "In_USD"].sum()
apac_total = apac_usd.loc[apac_usd["Date"].notna(), "In_USD"].sum()

# Calculate global revenue
global_total = us_total + eu_total + apac_total

# Display results
print("US Revenue:", round(us_total, 2))
print("EU Revenue:", round(eu_total, 2))
print("APAC Revenue:", round(apac_total, 2))
print("Total In USD Revenue:", round(global_total, 2))

US Revenue: 80483.55
EU Revenue: 89240.27
APAC Revenue: 111188.19
Total In USD Revenue: 280912.01


In [17]:
# Review surrounding EUR exchange rates
rates[(rates["currency"] == "EUR") & (rates["Date"].between("2026-02-16", "2026-02-22"))].rename(
    columns={"rate_date": "Rate Date", "currency": "Currency", "usd_per_unit": "USD Per Unit"}
)

,Rate Date,Currency,USD Per Unit,Date
183,2026-02-16,EUR,1.07607,2026-02-16
187,2026-02-17,EUR,1.07759,2026-02-17
191,2026-02-18,EUR,1.07595,2026-02-18
195,2026-02-19,EUR,1.07752,2026-02-19
196,2026-02-19,EUR,1.20682,2026-02-19
200,2026-02-20,EUR,1.07959,2026-02-20
204,2026-02-21,EUR,1.08062,2026-02-21
208,2026-02-22,EUR,1.08264,2026-02-22


In [18]:
# Review surrounding JPY exchange rates
rates[(rates["currency"] == "JPY") & (rates["Date"].between("2026-08-02", "2026-08-08"))].rename(
    columns={"rate_date": "Rate Date", "currency": "Currency", "usd_per_unit": "USD Per Unit"}
)

,Rate Date,Currency,USD Per Unit,Date
848,2026-08-02,JPY,0.006673,2026-08-02
852,2026-08-03,JPY,0.006686,2026-08-03
855,2026-08-04,JPY,0.006680,2026-08-04
859,2026-08-05,JPY,0.007459,2026-08-05
860,2026-08-05,JPY,0.006660,2026-08-05
864,2026-08-06,JPY,0.006673,2026-08-06
868,2026-08-07,JPY,0.006700,2026-08-07
872,2026-08-08,JPY,0.006690,2026-08-08


In [19]:
# Remove incorrect duplicate exchange rates
clean_rates = rates.copy()

clean_rates = clean_rates.drop(859)

clean_rates = clean_rates[
    ~((clean_rates["currency"] == "EUR") &
      (clean_rates["Date"] == "2026-02-19") &
      (clean_rates["usd_per_unit"] == 1.206820))
]

In [20]:
# Check remaining duplicate exchange rates
clean_rates.duplicated(subset=["Date", "currency"]).sum()

np.int64(0)

In [21]:
# Fill missing daily exchange rates
clean_rates = clean_rates.pivot(
    index="Date",
    columns="currency",
    values="usd_per_unit"
)

clean_rates = clean_rates.reindex(
    pd.date_range("2026-01-01", "2026-09-30")
)

clean_rates = clean_rates.interpolate()

clean_rates = clean_rates.stack().reset_index()

clean_rates.columns = ["Date", "currency", "usd_per_unit"]

In [22]:
# Correct abnormal exchange rates
clean_rates.loc[
    (clean_rates["Date"] == "2026-01-17") &
    (clean_rates["currency"] == "EUR"),
    "usd_per_unit"
] = 1.077415

clean_rates.loc[
    (clean_rates["Date"] == "2026-08-07") &
    (clean_rates["currency"] == "EUR"),
    "usd_per_unit"
] = 1.080390

clean_rates.loc[
    (clean_rates["Date"] == "2026-05-04") &
    (clean_rates["currency"] == "SGD"),
    "usd_per_unit"
] = 0.739360

In [23]:
# Check Clean Date and Raw Date formats in Consolidated Sales.csv
display(sales[["Order ID", "Raw Date", "Clean Date"]].head(10))
print("Missing Clean Date:", sales["Clean Date"].isna().sum())

,Order ID,Raw Date,Clean Date
0,AP-300436,08-06-2026,2026-06-08 00:00:00
1,AP-300635,09-08-2026,2026-08-09 00:00:00
2,AP-300707,02-09-2026,2026-09-02 00:00:00
3,AP-300763,19-09-2026,2026-09-19 00:00:00
4,AP-300067,23-01-2026,2026-01-23 00:00:00
5,AP-300085,30-01-2026,2026-01-30 00:00:00
6,AP-300248,05-04-2026,2026-04-05 00:00:00
7,AP-300192,07-03-2026,2026-03-07 00:00:00
8,AP-300299,21-04-2026,2026-04-21 00:00:00
9,AP-300184,03-03-2026,2026-03-03 00:00:00


Missing Clean Date: 0


In [24]:
# Create standardized Date column from Clean Date
sales["Date"] = pd.to_datetime(sales["Clean Date"])

# Correct incorrectly interpreted EU date
sales.loc[sales["Order ID"] == "EU-200336", "Date"] = pd.Timestamp("2026-04-12")

# Verify corrected date
sales.loc[sales["Order ID"] == "EU-200336", ["Order ID", "Raw Date", "Date"]]

,Order ID,Raw Date,Date
1463,EU-200336,12.04.2026,2026-04-12


In [25]:
# Identify duplicate order IDs
sales[sales.duplicated(subset=["Order ID"], keep=False)][
    ["Order ID", "Region", "Date", "City", "Product Name", "Qty", "Raw Local Amount"]
].sort_values("Order ID")

,Order ID,Region,Date,City,Product Name,Qty,Raw Local Amount
62,AP-300174,APAC,2026-03-02,Tokyo,Shelf Bracket Heavy Duty,16,19968.00
63,AP-300174,APAC,2026-03-02,Tokyo,Shelf Bracket Heavy Duty,16,19968.00
836,AP-300261,APAC,2026-04-11,Sydney,Locking Cam Latch,4,25.04
837,AP-300261,APAC,2026-04-11,Sydney,Locking Cam Latch,4,25.04
849,AP-300427,APAC,2026-06-04,Sydney,Locking Cam Latch,17,106.42
850,AP-300427,APAC,2026-06-04,Sydney,Locking Cam Latch,17,106.42
645,AP-300452,APAC,2026-06-13,Sydney,Cabinet Pull - Brushed Nickel,7,54.46
646,AP-300452,APAC,2026-06-13,Sydney,Cabinet Pull - Brushed Nickel,7,54.46
1309,EU-200289,EU,2026-04-01,Berlin,Corner Bracket 3in,7,21.77
1310,EU-200289,EU,2026-04-01,Berlin,Corner Bracket 3in,7,21.77


In [26]:
# Drop exact within-region duplicates (ignoring Index)
cols_to_check = [c for c in sales.columns if c != "Index"]
sales = sales.drop_duplicates(subset=cols_to_check).copy()

# Identify cross-region duplicate order IDs
duplicate_ids = sales.loc[sales["Order ID"].duplicated(keep=False), "Order ID"]

# Remove suspected duplicate EU transactions
sales = sales[~((sales["Region"] == "EU") & (sales["Order ID"].isin(duplicate_ids)))].copy()

# Check remaining duplicate order IDs
sales["Order ID"].duplicated().sum()

np.int64(0)

In [27]:
# Correct inconsistent tax percentages
sales.loc[sales["Tax Percentage"].between(0.01, 0.99), "Tax Percentage"] *= 100

# Assign regional tax percentages
sales.loc[sales["Region"] == "US", "Tax Percentage"] = 0
sales.loc[sales["Region"] == "EU", "Tax Percentage"] = 20

In [28]:
# Review APAC tax rates by currency
sales[sales["Region"] == "APAC"].groupby(
    ["Currency", "Tax Percentage"]
).size()

Currency  Tax Percentage
AUD       0.0                 2
          10.0              208
JPY       10.0              127
SGD       0.0                 1
          9.0               163
          10.0                5
dtype: int64

In [29]:
# Inspect the 3 APAC rows with 0% tax
sales[(sales["Region"] == "APAC") & (sales["Tax Percentage"] == 0)]

,City,Qty,Currency,Region,Index,Order ID,Raw Date,Clean Date,Product Name,Raw Local Amount,Local Unit Price,Tax Percentage,Date
151,Sydney,12,AUD,APAC,80,AP-300121,14-02-2026,2026-02-14 00:00:00,Drawer Slide 22in,383.76,31.98,0.0,2026-02-14
729,Singapore,30,SGD,APAC,371,AP-300588,22-07-2026,2026-07-22 00:00:00,Cabinet Pull - Matte Black,216.60,7.22,0.0,2026-07-22
824,Melbourne,4,AUD,APAC,128,AP-300196,07-03-2026,2026-03-07 00:00:00,Locking Cam Latch,25.04,6.26,0.0,2026-03-07


In [30]:
# Calculate revenue excluding taxes
sales["Net_Local_Revenue"] = sales["Raw Local Amount"] / (
    1 + sales["Tax Percentage"] / 100
)

In [31]:
# Match corrected exchange rates
final_sales = sales.merge(
    clean_rates,
    left_on=["Date", "Currency"],
    right_on=["Date", "currency"],
    how="left"
)

# Rename exchange rate column and set USD exchange rate to 1
final_sales = final_sales.rename(columns={"usd_per_unit": "USD Per Unit"})
final_sales.loc[final_sales["Currency"] == "USD", "USD Per Unit"] = 1

# Calculate net revenue in USD
final_sales["Net_Revenue_USD"] = (
    final_sales["Net_Local_Revenue"] * final_sales["USD Per Unit"]
)

In [32]:
# Check for missing values
print(final_sales[["Date", "Tax Percentage", "USD Per Unit", "Net_Revenue_USD"]].isna().sum())

# Check dates outside the reporting period
print("Out-of-period dates:", (~final_sales["Date"].between("2026-01-01", "2026-09-30")).sum())

Date               0
Tax Percentage     0
USD Per Unit       4
Net_Revenue_USD    4
dtype: int64
Out-of-period dates: 6


In [33]:
# Display net revenue by region
final_sales.groupby("Region")["Net_Revenue_USD"].sum().round(2)

Region
APAC    102699.74
EU       74082.51
US       80694.66
Name: Net_Revenue_USD, dtype: float64

In [34]:
# Save excluded transactions for documentation
excluded_sales = final_sales[
    ~final_sales["Date"].between("2026-01-01", "2026-09-30")
].copy()

# Keep transactions within the reporting period
final_sales = final_sales[
    final_sales["Date"].between("2026-01-01", "2026-09-30")
].copy()

# Check remaining missing exchange rates
print(final_sales["USD Per Unit"].isna().sum())

0


In [35]:
# Calculate corrected gross USD revenue
final_sales["Gross_USD"] = final_sales["Raw Local Amount"] * final_sales["USD Per Unit"]
corrected_gross = final_sales["Gross_USD"].sum()

In [36]:
# Calculate tax adjustments by region
final_sales["Tax_Adjustment"] = (final_sales["Net_Revenue_USD"] - final_sales["Gross_USD"])
eu_tax = final_sales.loc[final_sales["Region"] == "EU", "Tax_Adjustment"].sum()
apac_tax = final_sales.loc[final_sales["Region"] == "APAC", "Tax_Adjustment"].sum()

In [37]:
# Compare original and corrected exchange rates
comparison = final_sales.merge(
    original_rates[["Date", "currency", "usd_per_unit"]].rename(
        columns={"usd_per_unit": "usd_per_unit_original"}
    ),
    left_on=["Date", "Currency"],
    right_on=["Date", "currency"],
    how="left"
)

comparison.loc[comparison["Currency"] == "USD", "usd_per_unit_original"] = 1

# Calculate exchange rate adjustments
rate_adjustment = (
    comparison["Raw Local Amount"] * (comparison["USD Per Unit"] - comparison["usd_per_unit_original"])
).sum()

missing_adjustment = comparison.loc[comparison["usd_per_unit_original"].isna(), "Gross_USD"].sum()

print("Exchange Rate Corrections:", round(rate_adjustment, 2))
print("Missing Exchange Rates:", round(missing_adjustment, 2))

Exchange Rate Corrections: 82.99
Missing Exchange Rates: 2203.34


In [38]:
# Calculate remaining data adjustments
data_adjustment = corrected_gross - global_total - rate_adjustment - missing_adjustment

print("Data Adjustments:", round(data_adjustment, 2))


Data Adjustments: -988.34


In [39]:
# Create revenue bridge
bridge = pd.DataFrame({
    "Description": ["Original Raw Revenue in USD", "Data Adjustments",
                    "Exchange Rate Corrections", "Missing Exchange Rates",
                    "EU VAT Adjustment", "APAC Sales Tax Adjustment",
                    "Final Net USD Revenue"],
    "Amount_USD": [global_total, data_adjustment, rate_adjustment,
                   missing_adjustment, eu_tax, apac_tax,
                   final_sales["Net_Revenue_USD"].sum()]
})

bridge.round(2)

,Description,Amount_USD
0,Original Raw Revenue in USD,280912.01
1,Data Adjustments,-988.34
2,Exchange Rate Corrections,82.99
3,Missing Exchange Rates,2203.34
4,EU VAT Adjustment,-14816.50
5,APAC Sales Tax Adjustment,-10045.78
6,Final Net USD Revenue,257347.71


In [40]:
# Inspect product_master columns and values
print("Product Master Columns:", products.columns.tolist())
display(products)

Product Master Columns: ['product_id', 'product_name', 'category']


,product_id,product_name,category
0,P-101,Adjustable Shelf Pin Set,Hardware
1,P-102,Cabinet Hinge Set,Hinges
2,P-103,Cabinet Pull - Brushed Nickel,Pulls
3,P-104,Cabinet Pull - Matte Black,Pulls
4,P-105,Corner Bracket 3in,Brackets
5,P-106,Drawer Slide 18in,Slides
6,P-107,Drawer Slide 22in,Slides
7,P-108,Locking Cam Latch,Hardware
8,P-109,Shelf Bracket Heavy Duty,Brackets
9,P-110,Soft-Close Hinge,Hinges


In [41]:
# See rows where Product Name is not in product_master
bad_products = final_sales[~final_sales["Product Name"].isin(products["product_name"])]
bad_products[["Order ID", "Region", "Product Name", "Qty", "Local Unit Price", "Net_Revenue_USD"]]

,Order ID,Region,Product Name,Qty,Local Unit Price,Net_Revenue_USD
229,US-100559,US,Drawer Slide 2in,3,19.84,59.520000
491,AP-300781,APAC,Drawer Slide 26in,3,10.83,19.138087
1060,US-100109,US,Cabinet Pull - Chrome Polished,22,3.70,81.400000
1451,EU-200068,EU,Soft-Close Hinge Pro,5,NaN,48.887706


In [42]:
# Fix product name typos (adjust if you decide to exclude/flag any of the 4)
final_sales["Product Name"] = final_sales["Product Name"].replace({
    "Drawer Slide 2in": "Drawer Slide 22in",
    "Drawer Slide 26in": "Drawer Slide 22in",
    "Cabinet Pull - Chrome Polished": "Cabinet Pull - Brushed Nickel",
    "Soft-Close Hinge Pro": "Soft-Close Hinge"
})

In [43]:
# Add product_id and category from product_master
final_sales = final_sales.merge(
    products,
    left_on="Product Name",
    right_on="product_name",
    how="left"
).drop(columns=["product_name"])

# Verify 0 missing product IDs
final_sales["product_id"].isna().sum()

np.int64(0)

In [44]:
final_sales[final_sales["Qty"] <= 0][["Order ID", "Region", "Product Name", "Qty", "Raw Local Amount", "Net_Revenue_USD"]]

,Order ID,Region,Product Name,Qty,Raw Local Amount,Net_Revenue_USD
17,AP-300372,APAC,Drawer Slide 18in,-2,-4762.00,-29.000580
30,AP-300397,APAC,Soft-Close Hinge,-2,-4240.00,-25.833164
101,AP-300731,APAC,Cabinet Pull - Brushed Nickel,-3,-2265.00,-13.886509
123,AP-300782,APAC,Corner Bracket 3in,-3,-1380.00,-8.389145
228,US-100829,US,Drawer Slide 22in,-6,-119.04,-119.040000
265,US-100399,US,Drawer Slide 22in,-4,-75.60,-75.600000
280,US-100459,US,Drawer Slide 22in,-6,-113.40,-113.400000
289,AP-300408,APAC,Cabinet Hinge Set,-8,-139.28,-82.484148
311,AP-300449,APAC,Cabinet Hinge Set,-3,-45.48,-30.977721
312,AP-300521,APAC,Cabinet Hinge Set,-1,-15.16,-10.324934


In [45]:
# Export final results
final_sales.to_csv("Final_Consolidated_Sales.csv", index=False)
bridge.round(2).to_csv("Revenue_Bridge.csv", index=False)
excluded_sales.to_csv("Excluded_Transactions.csv", index=False)